# 应许之路 · 0 成本商用 3D 资产小样工坊（Microsoft TRELLIS · MIT 许可）

- **费用**：$0（使用 Google Colab 免费 T4 16GB GPU）
- **商用许可**：Microsoft TRELLIS 官方开源权重采用 **MIT License**（完全免费商用、零版税、无用户数限制、无需署名）。
- **隐私**：在个人 Colab 临时实例运行，不上传至任何公开社区画廊。
- **操作**：点击菜单 `代码执行程序 → 更改运行时类型 → T4 GPU`，然后依次运行下面 3 个单元格即可下载带完整 PBR 贴图的 `.glb` 文件。

In [ ]:
# 1. 安装 Microsoft TRELLIS (MIT) 及其依赖（约 2–3 分钟，仅首次需要）
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
%cd /content/TRELLIS
!pip install -q pillow imageio imageio-ffmpeg tqdm easydict opencv-python-headless scipy ninja rembg onnxruntime trimesh xatlas pyvista pymeshfix igraph transformers
!pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
!pip install -q kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html || true
!pip install -q nvdiffrast || pip install -q git+https://github.com/NVlabs/nvdiffrast.git
!pip install -q spconv-cu121
print('TRELLIS_ENV_READY')

In [ ]:
# 2. 上传你本地 design/review-slice-01/inputs/ 里的单图（如 abram_front.png 或 prop_amphora_ref.png）
from google.colab import files
uploaded = files.upload()
INPUT_FILES = list(uploaded.keys())
print('已上传:', INPUT_FILES)

In [ ]:
# 3. 一键生成带完整 PBR 贴图与自动减面的商用 .glb 并自动下载到本地
import os
os.environ['SPCONV_ALGO'] = 'native'
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
from google.colab import files

pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()

for fname in INPUT_FILES:
    stem = os.path.splitext(fname)[0]
    img = Image.open(fname).convert('RGBA')
    outputs = pipeline.run(img, seed=42)
    glb = postprocessing_utils.to_glb(
        outputs['gaussian'][0],
        outputs['mesh'][0],
        simplify=0.95,
        texture_size=1024,
    )
    out_glb = f'/content/{stem}_trellis_pbr.glb'
    glb.export(out_glb)
    print('已导出:', out_glb, os.path.getsize(out_glb), 'bytes')
    files.download(out_glb)